# 08 — Donchian turtle (daily USDT-M, top-100 universe)

Research only. **Does not dispatch live alerts or retune TEMA `W_*`.**

Mechanics (``scanner/donchian_turtle.py``):

- Entry: close &gt; **55-day** Donchian upper (prior bars only) + optional **20-day VWAP**
- Exit: close &lt; **20-day** Donchian lower
- Portfolio: up to **20** equal-weight names from a **top-100** liquidity universe

Data: Binance Vision USDT-M daily (``backtest.data_loader.load_tf_ohlcv``). First run downloads/cache months.


In [ ]:
import sys
from pathlib import Path
from datetime import date

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
elif (ROOT / "python" / "scanner").exists():
    ROOT = ROOT / "python"
sys.path.insert(0, str(ROOT))
print("python root:", ROOT)


In [ ]:
from backtest.donchian_turtle import (
    apply_split_metrics,
    benchmark_btc_returns,
    resolve_universe,
    run_turtle_portfolio,
)
from scanner.donchian_turtle import TurtleParams
import pandas as pd

START = date(2024, 1, 1)
END = date(2025, 8, 1)
SPLIT = date(2025, 1, 1)
TOP_UNIVERSE = 100
MAX_POSITIONS = 20

universe, src = resolve_universe(TOP_UNIVERSE)
print(f"Universe: {len(universe)} symbols ({src})")
print("head:", universe[:10])


In [ ]:
params = TurtleParams(entry_channel=55, exit_channel=20, use_vwap_filter=True)
result = run_turtle_portfolio(
    universe,
    START,
    END,
    params=params,
    max_positions=MAX_POSITIONS,
    rebalance_days=1,
)
result.universe_source = src
apply_split_metrics(result, SPLIT)

print("Symbols with data:", len(result.universe))
print("Full sample:", result.metrics)
print("In-sample (< split):", result.is_metrics)
print("Out-of-sample (>= split):", result.oos_metrics)


In [ ]:
btc = benchmark_btc_returns(START, END)
btc_metrics = {
    "total_return_pct": round(((1 + btc).prod() - 1) * 100, 3) if len(btc) else 0,
    "sharpe": round(btc.mean() / btc.std() * (365 ** 0.5), 3) if btc.std() > 0 else 0,
}
print("BTC buy-hold (daily):", btc_metrics)

eq = result.equity_curve
if not eq.empty:
    ax = eq["equity"].plot(figsize=(10, 4), title="Donchian turtle book equity (USD notional 100k)")
    ax.set_ylabel("equity")
    ax.figure.tight_layout()


## CLI (same engine)

```bash
cd python
python -m backtest.donchian_turtle --start 2024-01-01 --split 2025-01-01 --top-universe 100
```

Artifacts: ``backtest/results/donchian_turtle/summary.json`` and ``equity_curve.parquet``.
